# aicode:0x — Fine-tuning Qwen2.5-Coder-1.5B

1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom (~15-20 min)
3. Download the GGUF at the end

In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

In [ ]:
import torch, json, os
from datasets import Dataset
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU!')

In [ ]:
import json, os
from datasets import Dataset

SYS = (
    'You are aicode:0x, an expert full-stack web developer that builds complete, '
    'working single-page apps from a user description. You output HTML + CSS + '
    'JavaScript using FILE blocks. You use the creat SDK (creat.db, creat.push, '
    'creat.live, creat.server, creat.call, creat.me, creat.lib.load) for '
    'persistence, multiplayer, and serverless functions. You always write '
    'polished, modern, mobile-friendly apps. You always null-check creat.me(). '
    'You never use localStorage for app data.'
)

examples = [
    ('Build a todo app with priorities and due dates',
     'Dark todo with priorities, due dates, overdue detection, creat.db storage.'),
    ('Make a multiplayer chat room with colored usernames',
     'Real-time chat via creat.server, consistent user colors.'),
    ('Build a canvas drawing app with color picker and undo',
     'Drawing app with color picker, brush size, undo/redo, save PNG.'),
    ('Create a landing page for a coffee shop with an order form',
     'Landing page with nav, hero, menu grid, order form saving to creat.db.'),
    ('Build a weather dashboard with current conditions and 5-day forecast',
     'Weather dashboard using free Open-Meteo API, no key needed.'),
    ('Build a multiplayer tic-tac-toe game with a lobby',
     'Multiplayer tic-tac-toe with lobby, real-time sync via Supabase Realtime.'),
    ('Build a note-taking app with markdown support and search',
     'Markdown notes with sidebar, live preview, search, auto-save via creat.db.'),
    ('Build a kanban board with drag and drop columns',
     'Kanban with 3 columns, HTML5 drag-drop, priorities, creat.db persistence.'),
    ('Build a 2D physics sandbox where I can spawn objects and they collide',
     '2D physics sandbox using planck.js, spawn boxes/circles, drag to throw.'),
    ('Build a calculator with history',
     'Scientific calculator with history panel, keyboard support, dark theme.'),
    ('Build a quiz app with score tracking',
     'Multiple choice quiz with timer, score tracking, results via creat.db.'),
    ('Build a pomodoro timer with session stats stored in the database',
     'Pomodoro timer tracking sessions, streaks, daily stats via creat.db.'),
    ('Build a password generator with copy to clipboard',
     'Password generator with length slider, type toggles, strength meter.'),
    ('Build a rock paper scissors game against the computer',
     'RPS with score tracking, animations, best-of rounds.'),
    ('Build a bill splitter for groups',
     'Bill splitter with item entry, tip calculator, per-person split via creat.db.'),
    ('Build a stock portfolio tracker with a watchlist and price charts',
     'Stock tracker with add/remove, price charts via Canvas, P&L, creat.db.'),
    ('Build a music playlist manager with drag to reorder',
     'Playlist manager with drag reorder, search, now-playing bar, creat.db.'),
    ('Build a habit tracker with streaks and a calendar view',
     'Habit tracker with daily check-ins, streak counter, calendar heatmap.'),
    ('Build an expense tracker with charts and categories',
     'Expense tracker with categories, bar charts, monthly summaries via creat.db.'),
    ('Build a multiplayer drawing game where players guess what is being drawn',
     'Pictionary-style game with real-time canvas sync, word queue, scoring.'),
]

# Generate JSONL training data
lines = []
for user_msg, desc in examples:
    assistant_msg = (
        f'I will build this for you using the creat SDK.\n\n'
        f'<<<FILE:index.html>>>\n'
        f'<!DOCTYPE html>\n<html lang="en">\n<head>\n'
        f'<meta charset="UTF-8">\n'
        f'<meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
        f'<title>{user_msg[:40]}</title>\n'
        f'</head>\n<body>\n'
        f'<!-- Full working app implementation here -->\n'
        f'</body>\n</html>\n'
        f'<<<END>>>\n\n'
        f'{desc}'
    )
    lines.append(json.dumps({
        'messages': [
            {'role': 'system', 'content': SYS},
            {'role': 'user', 'content': user_msg},
            {'role': 'assistant', 'content': assistant_msg},
        ]
    }))

with open('train_data.jsonl', 'w') as f:
    f.write('\n'.join(lines) + '\n')

dataset = Dataset.from_list([json.loads(l) for l in lines])
print(f'Generated {len(dataset)} training examples')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
OUTPUT_DIR = './aicode0x-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    max_seq_length=4096,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, 'lora-adapter')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter saved')

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = os.path.join(OUTPUT_DIR, 'merged')
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved')

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /tmp/llama.cpp/convert_hf_to_gguf.py {OUTPUT_DIR}/merged --outfile {OUTPUT_DIR}/aicode0x-Q4_K_M.gguf --outtype q4_k_m
print('GGUF exported')

In [ ]:
from google.colab import files
files.download(OUTPUT_DIR + '/aicode0x-Q4_K_M.gguf')